In [ ]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
from huggingface_hub import hf_hub_download
torch.set_float32_matmul_precision('high')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- download model ---

gat_ckpt_path = hf_hub_download(
    repo_id="Ksgk-fy/sorl",
    filename="ts-k4-v128.pt",
    repo_type="model",
    local_dir="./ckpt",  # saves to ./ckpt/ts-k4-v128.pt
)

gat_config = GATConfig.gpt_size("small", [BOS_TOKEN_ID+1, 128])
model = GAT(gat_config)

# ---- load GAT ckpt ----
def local_load_ckpt(model, ckpt_path):
    """load compiled ckpt locally"""
    model_ckpt = torch.load(ckpt_path, map_location="cpu")['model']
    clean_state_dict = {k.replace("_orig_mod.", ""): v for k, v in model_ckpt.items()}
    model.load_state_dict(clean_state_dict)
    return model

model = local_load_ckpt(model, gat_ckpt_path)
model = model.to(device)
K = 4

In [ ]:
from data.tinystory_local import TinyStoriesDataLoader, TinyStoriesDataLoader_v2, collect_rollout_statistics, AbstractionStatistics
from data.tinystory_local import visualize_dynamics
import tiktoken 

# TinyStories Dataset + GPT2 tokenizer
# -------------------------------------
num_stories = 100
max_len = 128
doc_len = max_len  + (max_len - 1) // K # <-- doc len contains abstract tokens
pad_shift = 2
loader = TinyStoriesDataLoader(num_stories=num_stories, max_len=max_len, chunk_size=K, device=device, split="validation", pad_shift=pad_shift)

batch_size = 8
memory_span = 1792
attn_blocksize = 1792
max_iterations = 2

# ---- stat collection ---
abs_stats = AbstractionStatistics(
    n_doc=num_stories,
    n_abs=doc_len - max_len,
    abs_vocab_size=model.vocab_sizes[1],
    device=model.device
)

# --- tokenizer --- 
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>']

In [ ]:
# Generate with SoRL trained on TinyStories Dataset
# -------------------------------------------------
from sorl.neo_utils import generate
from data.tinystory_local import visualize_interleaved_alignment    

min_temperature = 0.0
tokens, doc_ids = loader.get_batch(1)

# idx = tokens[:, :15].clone()
idx = torch.tensor(enc.encode("Chrismas is coming soon, ")).unsqueeze(0)

img_frames = []
for i in range(120): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, temperature=torch.tensor(min_temperature))
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    img = visualize_interleaved_alignment(idx, model, enc, K=K, max_chunks=8)
    img_frames.append(img)

# ---- save to gif ----
if len(img_frames) > 0:
    img_frames[0].save(
        'tiny-tiny-stories-generation.gif',
        save_all=True,
        append_images=img_frames[1:],
        duration=400,  # milliseconds sper frame
        loop=0  # 0 = infinite loop
    )
    print(f"Saved GIF with {len(img_frames)} frames")